# 47 · Recorded integration tests
Level L5 · Part 6 · professional · ~30 min · builds on 46

## Goal
Record Scout's real Gemini run once to a **cassette** with vcrpy, then replay it in a pytest test with no
key and no network, with the API key scrubbed from the file and every call replayed in its recorded order.

## The idea
Lesson 46's fake model tests the loop, but cannot catch a wrong request format, a missing thought
signature (lesson 35) or a renamed field: only the real API rejects those. Calling it in every test run
is slow, costs money, needs a key in CI and varies each time. **Record and replay** solves both: run the
test once against the real API and save every HTTP request and response to a file, the **cassette**.
After that, the test is served from the cassette. Before saving, you remove keys and cookies from it
(**secret scrubbing**), because the cassette goes into Git.

## Picture
```mermaid
flowchart LR
    T["test_scout_e2e.py"] --> L["run_agent + GeminiModel"]
    L -- "HTTP request" --> V{"vcrpy"}
    V -- "record (once, with a key)" --> G["Gemini API"]
    G --> S["scrub keys"] --> C[("cassettes/<br/>mars_day.yaml")]
    V -- "replay (every run, no key)" --> C
```

## Step by step
### 1. Setup
The notebook's own model call is recorded by the course's recorder, `llm_replay` (which is itself
built on vcrpy). Scout's loop and tools come from `../apps/46/`, which lesson 46 wrote; this lesson
writes its files to `../apps/47/`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("47_recorded-integration-tests")

import contextlib, io, json, os, re, subprocess, tempfile
from pathlib import Path
import vcr, yaml
from dotenv import load_dotenv

load_dotenv("../.env")                            # the key is only used when recording
assert Path("../apps/46/scout_loop.py").exists(), "run lesson 46 first: it writes ../apps/46/"
Path("../apps/47/cassettes").mkdir(parents=True, exist_ok=True)
sys.dont_write_bytecode = True                    # no __pycache__ in the apps folders
sys.path[1:1] = ["../apps/47", "../apps/46"]
print("vcrpy", vcr.__version__, "| folder ready:", Path("../apps/47").resolve().name)

LLM mode: replay (47_recorded-integration-tests.yaml)
vcrpy 8.3.0 | folder ready: 47


### 2. The real model as a file
pytest runs files, so the Gemini adapter from lesson 35 moves into `gemini_model.py`. Two things matter
for recording: the **model id is pinned** in the file, and **temperature is 0**, so a re-recording is as
close to the old one as the API allows. The question and system prompt live here too, in code.
The first half of the file holds the pinned settings, the tool schemas and the client.

In [2]:
%%writefile ../apps/47/gemini_model.py
"""Scout's Gemini adapter (lesson 35) as a file. Pinned model, temperature 0, prompts in code."""
import json, os
from google import genai
from google.genai import types
from scout_tools import TOOLS

MODEL = "gemini-3.5-flash-lite"                     # pinned: a new model means a new recording
QUESTION = "How many minutes long is a day on Mars?"
SYSTEM = ("You are Scout, a research assistant over the user's notes. Look facts up with search_notes "
          "and do every calculation with the calculator tool. Answer in one sentence.")
FULL_SCHEMAS = [
    {"name": "search_notes", "description": TOOLS["search_notes"].__doc__,
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "calculator", "description": TOOLS["calculator"].__doc__,
     "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
]
# No key in replay: the cassette answers, so a placeholder is enough.
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")

Overwriting ../apps/47/gemini_model.py


Next, `to_contents` turns Scout's messages into Gemini contents.

In [3]:
%%writefile -a ../apps/47/gemini_model.py

def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                            # keeps the thought signature
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

Appending to ../apps/47/gemini_model.py


Last, the model class: it builds the request config and calls the API.

In [4]:
%%writefile -a ../apps/47/gemini_model.py

class GeminiModel:
    def __init__(self, system: str = SYSTEM):
        self.system = system

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        decls = [types.Tool(function_declarations=[types.FunctionDeclaration(
            name=s["name"], description=s["description"], parameters_json_schema=s["parameters"]) for s in tools])]
        cfg = types.GenerateContentConfig(system_instruction=self.system, temperature=0, max_output_tokens=300,
                                          tools=decls, automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text, "raw": raw}

Appending to ../apps/47/gemini_model.py


### 3. One real run, by hand
Run Scout once with the real model, to see what the test will record: it searches the notes,
calculates and answers. This needs a key the first time; after that the course's recording replays it.

In [5]:
from gemini_model import FULL_SCHEMAS, QUESTION, SYSTEM, GeminiModel
from scout_loop import run_agent
from scout_tools import TOOLS

live = run_agent(QUESTION, GeminiModel(), TOOLS, FULL_SCHEMAS)
print("answer:", live["answer"])

step 1: search_notes({'query': 'Mars day minutes'}) -> []
step 2: search_notes({'query': 'Mars'}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
step 3: calculator({'expression': '24 * 60 + 39'}) -> 1479
step 4: answer
answer: A day on Mars is approximately 1,479 minutes long (or 24 hours and 39 minutes).


This run searched twice ("Mars day minutes" found nothing, then "Mars"): four model calls. The
committed recording (step 6) is a separate run and took three. Even at temperature 0 a real model can
take a different path, so a live test cannot assert the exact steps, and each call costs tokens, time
and a key CI should never hold. A recorded test pins **one** real run.

### 4. The cassette settings, in one place
`vcr.VCR(...)` sets how recording works. **record_mode** `"none"` replays only and fails on any request
not in the cassette; `"all"` makes real calls and records them. **match_on** decides when a new request
"is" a recorded one: method, host, path (holds the model id) **and body** (prompt and history), so a
changed prompt cannot get an old answer. The `filter_*` options and the response hook scrub secrets.

In [6]:
%%writefile ../apps/47/recording.py
"""Cassette settings shared by the tests and the notebook."""
import os
from pathlib import Path
import vcr

CASSETTES = Path(__file__).parent / "cassettes"
SECRET_HEADERS = ["x-goog-api-key", "authorization", "x-api-key", "cookie", "set-cookie"]
SECRET_QUERY = ["key", "api_key"]

def drop_secret_headers(response: dict) -> dict:
    """Responses can carry cookies too; drop them before saving."""
    for name in list(response["headers"]):
        if name.lower() in SECRET_HEADERS:
            response["headers"].pop(name)
    return response

def make_vcr(record_mode: str | None = None) -> vcr.VCR:
    return vcr.VCR(
        cassette_library_dir=str(CASSETTES),
        record_mode=record_mode or os.environ.get("SCOUT_RECORD", "none"),   # CI default: replay only
        match_on=["method", "host", "path", "body"],
        filter_headers=SECRET_HEADERS,              # removed from every recorded request
        filter_query_parameters=SECRET_QUERY,       # ...and from the URL
        before_record_response=drop_secret_headers,
        decode_compressed_response=True,            # readable JSON in the file, not gzip bytes
    )

Overwriting ../apps/47/recording.py


### 5. The recorded test
The test wraps one Scout run in a cassette. It checks what a fake could not: the real API accepted
every request, Scout used the two tools in order, and the answer holds 1479. In replay it also checks
**all_played**: every recorded call was used, so a loop that suddenly makes fewer calls fails too. A
second test scans every cassette for the key. These assertions pin the recorded run: after
re-recording, if the model took a different path, review the cassette diff and then update the
expected tools and call count.

In [7]:
%%writefile ../apps/47/test_scout_e2e.py
"""End-to-end test: Scout with the real Gemini model, recorded once to cassettes/mars_day.yaml."""
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).parent.parent / "46"))   # Scout's loop and tools (lesson 46)
from recording import CASSETTES, make_vcr
from gemini_model import FULL_SCHEMAS, QUESTION, GeminiModel
from scout_loop import run_agent
from scout_tools import TOOLS

def test_scout_answers_mars_day_with_the_real_model():
    with make_vcr().use_cassette("mars_day.yaml", allow_playback_repeats=False) as cassette:
        result = run_agent(QUESTION, GeminiModel(), TOOLS, FULL_SCHEMAS)
    tools = [c["name"] for m in result["messages"] for c in m.get("tool_calls", [])]
    assert tools == ["search_notes", "calculator"]
    assert "1479" in result["answer"].replace(",", "")
    assert result["stop"] == "end_of_turn" and result["errors"] == 0
    assert len(cassette) == 3                                   # three model calls recorded
    if cassette.record_mode == "none":
        assert cassette.all_played                              # ...and all three replayed, in order

def test_cassettes_hold_no_secrets():
    key = os.environ.get("GOOGLE_CLOUD_API_KEY")
    for path in CASSETTES.glob("*.yaml"):
        text = path.read_text()
        assert "x-goog-api-key" not in text.lower(), f"{path.name} holds the key header"
        assert not key or key not in text, f"{path.name} holds the key"

Overwriting ../apps/47/test_scout_e2e.py


### 6. Record once
The pytest helper from lesson 46, with two additions: extra environment variables, and an option to
**remove every API key** from the test's environment. The record cell sets `SCOUT_RECORD=all`. It only
records when the course is re-recording (`TUT_LLM_MODE=record`) or no cassette exists yet; otherwise
it keeps the committed cassette, as a teammate would.

In [8]:
KEYS = ["GOOGLE_CLOUD_API_KEY", "GOOGLE_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY"]

def pytest(target="test_scout_e2e.py", folder="../apps/47", show=6, extra=None, no_keys=False) -> int:
    env = {k: v for k, v in os.environ.items() if not (no_keys and k in KEYS)}
    env |= {"PYTHONDONTWRITEBYTECODE": "1", "COLUMNS": "60", **(extra or {})}
    out = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no",
                          "--tb=line", target], cwd=folder, capture_output=True, text=True, env=env)
    lines = out.stdout.strip().splitlines()
    print("\n".join(re.sub(r" in \d+\.\d+s", "", line) for line in lines[-show:]))
    return out.returncode

CASSETTE = Path("../apps/47/cassettes/mars_day.yaml")
print("cassette committed already:", CASSETTE.exists())

cassette committed already: True


Now the recording cell. By default it keeps the committed cassette. With a key and
`TUT_LLM_MODE=record` (or no cassette yet) it sets `SCOUT_RECORD=all`, makes three real Gemini calls and
writes `mars_day.yaml` (the recorded run passed both tests).

In [9]:
if llm_replay.mode() == "record" or not CASSETTE.exists():
    CASSETTE.unlink(missing_ok=True)               # "all" would append to an old file
    print("recording, return code:", pytest(extra={"SCOUT_RECORD": "all"}))
else:
    print(f"{CASSETTE.name} is committed: keeping it (TUT_LLM_MODE=record records it again)")

mars_day.yaml is committed: keeping it (TUT_LLM_MODE=record records it again)


### 7. Look inside the cassette
A cassette is plain YAML: a list of `interactions`, each one request and its response. We print each
request's path, what the model replied (a tool call or text), and the request headers that were kept.

In [10]:
tape = yaml.safe_load(CASSETTE.read_text())["interactions"]
for i, it in enumerate(tape, 1):
    parts = json.loads(it["response"]["body"]["string"])["candidates"][0]["content"]["parts"]
    said = [p["functionCall"]["name"] if "functionCall" in p else "text" for p in parts if "functionCall" in p or "text" in p]
    print(i, it["request"]["method"], it["request"]["uri"].split("/models/")[-1], it["response"]["status"]["code"], said)
print("request headers kept:", sorted(tape[0]["request"]["headers"]))
print("key header in file:", "x-goog-api-key" in CASSETTE.read_text().lower())

1 POST gemini-3.5-flash-lite:generateContent 200 ['search_notes']
2 POST gemini-3.5-flash-lite:generateContent 200 ['calculator']
3 POST gemini-3.5-flash-lite:generateContent 200 ['text']
request headers kept: ['accept', 'accept-encoding', 'connection', 'content-length', 'content-type', 'host', 'user-agent', 'x-goog-api-client']
key header in file: False


Three calls, in order: the model asked for `search_notes`, then `calculator`, then answered. The model
id is in the path, so it is part of every match. The kept headers are harmless; the key header is gone.

### 8. Replay with no key
This is the run CI does on every commit: `SCOUT_RECORD` unset (so `"none"`), and **no API key** in the
environment. If any request were not in the cassette, vcrpy would raise instead of calling the network.

In [11]:
code = pytest(no_keys=True)
print("return code:", code)

..                                                   [100%]
2 passed
return code: 0


Both tests pass from the cassette with no key, against the same bytes the real API sent.

### 9. Fixed call order, in the notebook
The same settings work outside pytest. Replay prints the loop's steps; `play_count` shows how many
recorded calls were used. With `allow_playback_repeats=False` each recorded call is served **once**,
in order, so the replay is exactly the recorded conversation.

In [12]:
from recording import make_vcr
with make_vcr("none").use_cassette("mars_day.yaml", allow_playback_repeats=False) as cassette:
    replayed = run_agent(QUESTION, GeminiModel(), TOOLS, FULL_SCHEMAS)
print("answer:", replayed["answer"])
print("played", cassette.play_count, "of", len(cassette), "| all played:", cassette.all_played)

step 1: search_notes({'query': 'Mars'}) -> [{"id": 1, "text": "Mars has two small moons, Phobos and Deimos."}, {"
step 2: calculator({'expression': '24 * 60 + 39'}) -> 1479
step 3: answer
answer: A day on Mars is 1,479 minutes long.
played 3 of 3 | all played: True


## At work
- **Three layers of tests**: fake-model tests on every commit (lesson 46), cassette tests for the real
  wire format with no key, and evals for answer quality, which need fresh real calls → lesson 48.
- **Cassettes go stale.** A vendor changes a field or retires the pinned model, and old cassettes keep
  passing. Teams re-record on a schedule → lesson 53.
- **Cassettes are committed files**: a key or customer data in them leaks to everyone with repo
  access. Scrub on record and scan in CI → lesson 50.
- **A failing run becomes a test**: log it, replay it as a fixed script → lesson 49.

## What just happened
- `gemini_model.py` holds the real adapter with a pinned model, temperature 0 and prompts in code.
- `recording.py` sets the cassette rules once: replay-only by default, match on method, host, path and
  body, and drop key headers, key query parameters and cookies.
- One run with `SCOUT_RECORD=all` saved three real calls to `cassettes/mars_day.yaml`, with no key in it.
- The same test then passed with no key, from the cassette; replay in the notebook used all three
  recorded calls, in order.

## Common mistakes
**1. Recording without filters.** vcrpy saves every request header by default, API key included. Here
a stand-in API built with httpx2's `MockTransport` (the HTTP client the anthropic and openai SDKs use), so no network is
needed, receives a fake key.

In [13]:
import httpx2
FAKE_KEY = "demo-key-not-real"
stand_in = httpx2.Client(transport=httpx2.MockTransport(lambda req: httpx2.Response(200, json={"text": "hi"})))
tmp = Path(tempfile.mkdtemp())
with vcr.VCR(record_mode="all").use_cassette(str(tmp / "leaky.yaml")):     # no filters
    stand_in.post("https://example.com/v1/generate", headers={"x-goog-api-key": FAKE_KEY}, json={"q": "hi"})
assert FAKE_KEY not in (tmp / "leaky.yaml").read_text(), "the API key was written to the cassette"

AssertionError: the API key was written to the cassette

Fix: record through `make_vcr`, whose filters drop the header before the file is written.

In [14]:
with make_vcr("all").use_cassette(str(tmp / "clean.yaml")):
    stand_in.post("https://example.com/v1/generate", headers={"x-goog-api-key": FAKE_KEY}, json={"q": "hi"})
print("key in clean cassette:", FAKE_KEY in (tmp / "clean.yaml").read_text())

key in clean cassette: False


**2. Changing the prompt and keeping the old cassette.** Someone adds "Be brief." to the system prompt.
If the matcher ignored the body, the test would replay answers to the **old** prompt and pass. Matching
on the body makes it fail at the first request.

In [15]:
from vcr.errors import CannotOverwriteExistingCassetteException
try:
    with make_vcr("none").use_cassette("mars_day.yaml", allow_playback_repeats=False):
        with contextlib.redirect_stdout(io.StringIO()):
            run_agent(QUESTION, GeminiModel(SYSTEM + " Be brief."), TOOLS, FULL_SCHEMAS)
except CannotOverwriteExistingCassetteException as e:
    msg = re.sub(r"\('[^)]*mars_day\.yaml'\)", "('cassettes/mars_day.yaml')", str(e))
    lines = [l for l in msg.splitlines() if l.strip()]
    i = next(i for i, l in enumerate(lines) if "Matchers failed" in l)
    print(type(e).__name__ + ":", *lines[:3], lines[i], lines[i + 1][:100], sep="\n")

CannotOverwriteExistingCassetteException:
Can't overwrite existing cassette ('cassettes/mars_day.yaml') in your current record mode (<RecordMode.NONE: 'none'>).
No match for the request (<Request (POST) https://aiplatform.googleapis.com/v1beta1/publishers/google/models/gemini-3.5-flash-lite:generateContent>) was found.
Found 3 similar requests with 1 different matcher(s) :
Matchers failed :
body - assertion failure :


Fix: when a prompt, model or config changes, **re-record** (`SCOUT_RECORD=all pytest test_scout_e2e.py`
with a key), and review the cassette's diff in the pull request like code. Then CI replays again.

In [16]:
print("committed prompt still replays:", pytest(no_keys=True, show=1) == 0)

2 passed
committed prompt still replays: True


## Production checklist
- [ ] CI runs cassette tests with `record_mode="none"` and **no** API key, so a missing recording fails
  instead of calling the API.
- [ ] Requests match on method, host, path **and body**; repeats are off, and replay checks `all_played`.
- [ ] Key headers, key query parameters and cookies are filtered on record; a test scans every cassette.
- [ ] Model id pinned and prompts in code.
- [ ] Re-record on a schedule and on every model or prompt change; review cassette diffs in PRs.
- [ ] No customer data in recorded prompts, or scrub bodies with `before_record_request`.

## Try it
Write `../apps/47/test_try_it.py` with one test that replays `mars_day.yaml` (no new recording, no key)
and checks the tool results the model saw: the calculator ran exactly once, without error, and returned
`1479`. Use `make_vcr()` and the same imports as `test_scout_e2e.py`. Run it with
`pytest("test_try_it.py", no_keys=True)`.

In [17]:
# your code here

Close the notebook's recorder.

In [18]:
llm_replay.stop()

## Key terms
- **record and replay** — saving a real API's HTTP traffic once and serving later runs from the copy.
- **cassette** — vcrpy's file (YAML here) of recorded requests and responses.
- **record mode** — when real calls happen: `"none"` only replays, `"all"` records all, `"once"` records if the file is missing.
- **request matching** — the fields (`match_on`) that pick which recorded response answers a request.
- **secret scrubbing** — removing keys, tokens and cookies from a recording before saving it.
- **stale cassette** — a recording that no longer matches the code or the live API.